# Stage 18 结果质量审查

## tl;dr

核心文件与算术通过；确认性推断需修复TASK_KEYED配对、crossed bootstrap、多重比较、成功吞吐、消融构念和协议覆盖。


## Context & Methods

### Key Assumptions

- 训练运行粒度：profile × condition × training seed。
- locked粒度：再增加evaluation profile；每个job有100个共享测试seed。
- 本notebook只读原始结果，不覆盖任何v1训练或测试文件。


In [1]:
from pathlib import Path
import json
from stage18_quality_audit_generator import build_audit
WORK_ROOT = Path('/Users/lab4099/Desktop/Mujoco/work')
audit = build_audit(WORK_ROOT)
print(json.dumps({
    'assessment': audit['overall_assessment'],
    'coverage': audit['coverage'],
    'multiplicity': audit['multiplicity'],
}, ensure_ascii=False, indent=2))


{
  "assessment": "NEEDS_REVISION_BEFORE_TRO_CONFIRMATORY_CLAIMS",
  "coverage": {
    "training_runs_expected": 120,
    "training_runs_present": 120,
    "training_run_completion_rate": 1.0,
    "history_rows": 240000,
    "training_episodes": 960000,
    "ppo_artifacts_expected": 5040,
    "ppo_artifacts_present": 5040,
    "locked_jobs_expected": 180,
    "locked_jobs_present": 180,
    "locked_job_unique_keys": 180,
    "policy_test_episodes": 18000,
    "baseline_test_episodes": 18000,
    "policy_test_tasks": 360000,
    "baseline_test_tasks": 360000,
    "official_comparisons": 36,
    "illegal_action_count": 0,
    "resource_leak_count": 0
  },
  "multiplicity": {
    "comparison_count": 36,
    "raw_p_lt_0_05": 17,
    "global_holm_p_lt_0_05": 0,
    "global_bh_fdr_q_lt_0_05": 16,
    "minimum_attainable_exact_two_sided_p_with_10_seeds": 0.001953125
  }
}


## Data

完整性与冻结协议覆盖检查。


In [2]:
print('Coverage rows:')
for row in audit['coverage_rows']:
    print(row)
print('Scenario coverage:')
for row in audit['scenario_coverage']:
    print(row)


Coverage rows:
{'area': '训练单元', 'expected': 120, 'actual': 120, 'status': 'PASS'}
{'area': 'PPO history updates', 'expected': 240000, 'actual': 240000, 'status': 'PASS'}
{'area': 'PPO checkpoint/final/best', 'expected': 5040, 'actual': 5040, 'status': 'PASS'}
{'area': 'Locked-test jobs', 'expected': 180, 'actual': 180, 'status': 'PASS'}
{'area': 'Policy test episodes', 'expected': 18000, 'actual': 18000, 'status': 'PASS'}
{'area': '统计比较', 'expected': 36, 'actual': 36, 'status': 'PASS'}
Scenario coverage:
{'scenario': 'MEDIUM', 'locked_test_present': True, 'status': 'COVERED'}
{'scenario': 'DENSE', 'locked_test_present': True, 'status': 'COVERED'}
{'scenario': 'CROSS_HEAVY', 'locked_test_present': False, 'status': 'MISSING'}
{'scenario': 'BURST', 'locked_test_present': True, 'status': 'COVERED'}
{'scenario': 'MIXED_CONTINUOUS_RECOVERY', 'locked_test_present': False, 'status': 'MISSING'}


## Results

统计校正、成功吞吐和跨负载描述性结果。


In [3]:
print('Full vs rule baseline:')
for row in audit['main_effects']:
    print({key: row[key] for key in ('profile','metric','mean_delta','ci_low','ci_high','raw_p','bh_q_global36','holm_p_global36')})
print('Successful-delivery throughput:')
for row in audit['successful_throughput']:
    print(row)


Full vs rule baseline:
{'profile': 'MEDIUM', 'metric': 'reward', 'mean_delta': 0.3520364408151257, 'ci_low': 0.22016890900882138, 'ci_high': 0.4860993935117085, 'raw_p': 0.001953125, 'bh_q_global36': 0.00439453125, 'holm_p_global36': 0.0703125}
{'profile': 'MEDIUM', 'metric': 'success_rate', 'mean_delta': 0.009600000000000004, 'ci_low': 0.008200000000000002, 'ci_high': 0.011051249999999988, 'raw_p': 0.001953125, 'bh_q_global36': 0.00439453125, 'holm_p_global36': 0.0703125}
{'profile': 'MEDIUM', 'metric': 'throughput_tasks_per_hour', 'mean_delta': -0.14450577758875438, 'ci_low': -0.16991514634003646, 'ci_high': -0.12137551378017097, 'raw_p': 0.001953125, 'bh_q_global36': 0.00439453125, 'holm_p_global36': 0.0703125}
{'profile': 'DENSE', 'metric': 'reward', 'mean_delta': 19.223568318263638, 'ci_low': 18.324318822081647, 'ci_high': 20.137260909169033, 'raw_p': 0.001953125, 'bh_q_global36': 0.00439453125, 'holm_p_global36': 0.0703125}
{'profile': 'DENSE', 'metric': 'success_rate', 'mean_del

In [4]:
print('Cross-load descriptive summary:')
for row in audit['cross_load']:
    print(row)


Cross-load descriptive summary:
{'training_profile': 'MEDIUM', 'evaluation_profile': 'MEDIUM', 'route': 'MEDIUM→MEDIUM', 'reward_delta': 0.35203644081512664, 'success_delta_pp': 0.9599999999999975, 'successful_throughput_delta': 0.3918837666435209, 'positive_reward_seed_count': 10, 'car_dog_car_share_pct': 21.634999999999998, 'single_dog_share_pct': 28.415000000000003}
{'training_profile': 'MEDIUM', 'evaluation_profile': 'DENSE', 'route': 'MEDIUM→DENSE', 'reward_delta': 7.3805959518862085, 'success_delta_pp': 1.0549999999999982, 'successful_throughput_delta': -1.5578458502179386, 'positive_reward_seed_count': 8, 'car_dog_car_share_pct': 20.365, 'single_dog_share_pct': 29.755}
{'training_profile': 'MEDIUM', 'evaluation_profile': 'BURST', 'route': 'MEDIUM→BURST', 'reward_delta': 16.622910835344527, 'success_delta_pp': 1.3400000000000034, 'successful_throughput_delta': -4.130192805775414, 'positive_reward_seed_count': 10, 'car_dog_car_share_pct': 19.785, 'single_dog_share_pct': 30.3499999

## Takeaways

1. 数据完整性PASS，确认性统计FAIL。
2. 当前主要结果可保留为探索性v1，但需要新鲜TASK_KEYED locked test。
3. 去持续位置是复合消融；队列与交接显式cue目前没有独立贡献证据。
4. 训练权重可继续使用；消融重构和新增算法基线才需要重新训练。
